<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/01_chunk_documents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!git clone https://github.com/yaranoun/ML-Tech.git

Cloning into 'ML-Tech'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 147 (delta 76), reused 6 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 57.81 KiB | 970.00 KiB/s, done.
Resolving deltas: 100% (76/76), done.


In [4]:
from pathlib import Path

%cd ML-Tech
raw_folder = Path("data/raw")

documents = []

for file_path in raw_folder.glob("*.txt"):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    documents.append({
        "filename": file_path.name,
        "content": text
    })

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(doc["filename"])

/content/ML-Tech
Loaded 6 documents
Personal attendance required.txt
Passport of adopted, born in special circumstances child or a citizen without a family name.txt
Certifying Passport.txt
Ex-porting Biometric Passport.txt
Lost Passport or Stolen Passport.txt
Biometric Passport.txt


In [5]:
document_sections = {
    "Biometric Passport Documents.txt": [
        "Requested documents:",
        "Remarks:",
        "Fees:",
        "NB:"
    ],

    "Lost Passport or Stolen Passport.txt": [
        "Lost Passport:",
        "Stolen Passport:",
        "NB:"
    ],

    "Ex-porting Biometric Passport.txt": [
        "Exporting a Lebanese passport",
        "Exporting a Foreign passport",
        "Lebanese or foreign passport shipped:",
        "For travel agencies that plan to ship passports:",
        "For the individual planning on shipping his passport with another traveler:",
        "Nb:"
    ],
    "Passport of an adopted, born in special circumstances child or a citizen without a family name.txt":[
        "The requested documents",
        "NB:",
        "Child born in special cirmustances",
        "A passport for a minor:",
        "A citizen without a family name"
    ],
    "Personal attendance required.txt":[
        "Personal attendance required",
        "Exemption from attendance",
        "Exemption from fees"
    ],
    "Certifying Passport.txt":[]
}

In [6]:
def remove_metadata(text):
    if "Content:" in text:
        return text.split("Content:", 1)[1].strip()
    return text.strip()

In [8]:
import re

def chunk_document(filename, text, headings):
    chunks = []

    # If there are no headings, keep the whole document
    if not headings:
        chunks.append({
            "document": filename,
            "section": "Full Document",
            "text": text.strip()
        })
        return chunks

    # Build a regex from the headings
    pattern = "|".join(re.escape(h) for h in headings)

    # Split while keeping the headings
    parts = re.split(f"({pattern})", text)

    current_heading = None
    current_text = ""

    for part in parts:

        if part in headings:

            if current_heading is not None:
                chunks.append({
                    "document": filename,
                    "section": current_heading,
                    "text": current_text.strip()
                })

            current_heading = part.rstrip(":")
            current_text = ""

        else:
            current_text += part

    # Save the last chunk
    if current_heading is not None:
        chunks.append({
            "document": filename,
            "section": current_heading,
            "text": current_text.strip()
        })

    return chunks

In [10]:
all_chunks = []

for document in documents:

    filename = document["filename"]
    text = document["content"]

    headings = document_sections.get(filename, [])

    chunks = chunk_document(filename, text, headings)

    all_chunks.extend(chunks)

In [11]:
for document in documents:
    filename = document["filename"]
    text = document["content"]

    # remove URL, title, category, keywords, etc.
    clean_text = remove_metadata(text)

    headings = document_sections.get(filename, [])

    chunks = chunk_document(
        filename,
        clean_text,
        headings
    )

    all_chunks.extend(chunks)

In [12]:
import json

with open("data/processed/chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=4, ensure_ascii=False)